In [1]:
import logging
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn


from vasae.data.dataset import GPT2LayerActivations
from vasae.configs.data import DataConfig
from vasae.models.factory import BlackBoxModelConfig, load_embeding_layer

logging.basicConfig(
    format="[%(levelname)s] %(asctime)s %(message)s",
    level=logging.INFO,
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger()

In [2]:
class CFG:
    seed = 42

    meta_path = "/scratch/b5bq/pu22650.b5bq/activations_gpt2_Geralt-Targaryen_openwebtext2/meta.json"
    layer_name = "transformer.h.5"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = "gpt2"
    save_dir = "out"



def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [7]:



data_cfg = DataConfig(
    train_batchsize=10000,
    layer_name="transformer.h.1",
    data_dir=Path(r"/scratch/b5bq/pu22650.b5bq/activations_gpt2_Geralt-Targaryen_openwebtext2"),
)

In [4]:


blackbox_model_cfg = BlackBoxModelConfig(
    name="gpt2",
    dir=Path(r"/scratch/b5bq/pu22650.b5bq/VASAE_out/BlackBoxModels/gpt2"),
)

In [5]:
args = CFG()
set_seed(args.seed)
dataset = GPT2LayerActivations(data_cfg)


In [8]:
from vasae.data.dataset import get_dataloader


train_loader, valid_loader, test_loader = get_dataloader(
        data_cfg, 42
    )

In [10]:
train_act = next(iter(train_loader))["activations"]

In [11]:
train_act.shape

torch.Size([10000, 64, 768])

In [12]:
H = train_act.flatten(0, 1)    

In [13]:
H.shape

torch.Size([640000, 768])

In [14]:
idx = torch.randperm(H.size(0), device=H.device)[:1024]
H_sample = H[idx]  # [1024, 768]

In [6]:
dataset[0:len(dataset)]['activations'].shape

TypeError: list indices must be integers or slices, not str

/home/b5bq/pu22650.b5bq/miniforge3/envs/qwen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:



emb = load_embeding_layer(blackbox_model_cfg)

In [20]:
E = emb.weight

In [16]:
import torch
from typing import Tuple

@torch.no_grad()
def projection_error_spanE(H: torch.Tensor, E: torch.Tensor) -> torch.Tensor:
    """
    Compute mean ||h - P h||^2 where P projects onto span of rows of E (i.e., span of token embeddings in R^d).
    H: [N, d]
    E: [V, d]
    Returns: scalar tensor (mean squared L2 error)
    """
    assert H.dim() == 2 and E.dim() == 2
    assert H.size(1) == E.size(1)
    device = H.device
    E = E.to(device)

    # SVD of E: E = U S Vh, row-space basis is V = Vh^T
    # rank r = number of non-trivial singular values
    U, S, Vh = torch.linalg.svd(E, full_matrices=False)  # S: [d], Vh: [d, d]
    eps = 1e-6 * S.max()
    r = int((S > eps).sum().item())
    V = Vh.transpose(0, 1)[:, :r]  # [d, r], orthonormal basis

    # Projection: P(h) = V V^T h  -> compute as (H @ V) @ V^T
    H_proj = (H @ V) @ V.transpose(0, 1)  # [N, d]
    err = (H - H_proj).pow(2).sum(dim=1).mean()
    return err


@torch.no_grad()
def streaming_topM_candidates(
    H: torch.Tensor,
    E: torch.Tensor,
    M: int = 1024,
    chunk: int = 4096,
    use_abs: bool = True,
) -> torch.Tensor:
    """
    For each h, find top-M candidate token indices by (abs) dot-product with embeddings.
    Avoids materializing [N,V] all at once by scanning vocab in chunks.
    Returns: idx [N, M] (token indices)
    """
    assert H.dim() == 2 and E.dim() == 2
    N, d = H.shape
    V, d2 = E.shape
    assert d == d2

    device = H.device
    E = E.to(device)

    # maintain running topM
    best_vals = torch.full((N, M), -float("inf"), device=device)
    best_idx  = torch.zeros((N, M), dtype=torch.long, device=device)

    for start in range(0, V, chunk):
        end = min(V, start + chunk)
        Ec = E[start:end]  # [c, d]
        scores = H @ Ec.transpose(0, 1)  # [N, c]
        if use_abs:
            scores = scores.abs()

        vals, idx = torch.topk(scores, k=min(M, end-start), dim=1)  # [N, k']
        idx = idx + start

        merged_vals = torch.cat([best_vals, vals], dim=1)  # [N, M+k']
        merged_idx  = torch.cat([best_idx,  idx], dim=1)

        new_vals, new_pos = torch.topk(merged_vals, k=M, dim=1)
        new_idx = merged_idx.gather(1, new_pos)

        best_vals, best_idx = new_vals, new_idx

    return best_idx


@torch.no_grad()
def omp_k_error(
    H: torch.Tensor,
    E: torch.Tensor,
    k: int = 8,
    M: int = 1024,
    candidate_chunk: int = 4096,
    use_abs_candidates: bool = True,
    use_abs_select: bool = True,
) -> torch.Tensor:
    """
    Approximate oracle error via OMP@k on a per-sample candidate set.
    Steps:
      1) for each h, pick top-M candidates by (abs) dot product
      2) run OMP selection of k atoms among those M
      3) solve least squares over selected atoms
    Returns: mean ||h - h_hat||^2
    """
    assert H.dim() == 2 and E.dim() == 2
    N, d = H.shape
    V, d2 = E.shape
    assert d == d2

    device = H.device
    E = E.to(device)

    cand_idx = streaming_topM_candidates(
        H, E, M=M, chunk=candidate_chunk, use_abs=use_abs_candidates
    )  # [N, M]

    errs = []
    # OMP is hard to fully vectorize; do mini-batches to keep it manageable.
    # You can set N smaller (e.g., sample 1024 activations per layer) for speed.
    for n in range(N):
        h = H[n]  # [d]
        idx = cand_idx[n]  # [M]
        D = E.index_select(0, idx)  # [M, d], atoms

        r = h.clone()
        selected = []

        for _ in range(k):
            corr = D @ r  # [M]
            if use_abs_select:
                j = int(torch.argmax(corr.abs()).item())
            else:
                j = int(torch.argmax(corr).item())
            if j in selected:
                break
            selected.append(j)

            A = D[selected]  # [t, d]
            # Solve min_c || h - A^T c ||^2  (A^T: [d, t])
            # torch.linalg.lstsq expects (m,n) with m>=n typically; here m=d, n=t ok.
            sol = torch.linalg.lstsq(A.transpose(0, 1), h).solution  # [t]
            r = h - (A.transpose(0, 1) @ sol)  # [d]

        err = r.pow(2).sum()
        errs.append(err)

    return torch.stack(errs).mean()




In [15]:
H = H_sample

In [22]:
if __name__ == "__main__":
    # ---- Replace these with your actual loading ----
    # Example:
    # E = model.get_input_embeddings().weight.detach().cpu()  # [V, d]
    # H = torch.load("layer12_activations.pt")  # [N, d]
    #
    # For demonstration, make dummy tensors:
    V, d = 50000, 768
    N = 512  # 建议先用小 N 估计下界；比如每层抽样 512/1024
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(0)
    # E = torch.randn(V, d, device=device)
    # H = torch.randn(N, d, device=device)

    pe = projection_error_spanE(H, E)
    print("Projection span(E) mean error:", float(pe))

    omp = omp_k_error(H, E, k=8, M=1024, candidate_chunk=4096)
    print("OMP@8 approx-oracle mean error:", float(omp))

Projection span(E) mean error: 4.807401143835932e-09
OMP@8 approx-oracle mean error: 944.2407836914062
